<a href="https://colab.research.google.com/github/SCodezz/Naturalfuzz-pyspark/blob/main/naturalfuzz_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Captures core essence: branch profiling → seed selection → interleaving mutation → evaluation

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
import random

# Spark setup
spark = SparkSession.builder.master("local[*]").appName("NaturalFuzz_Simplified").getOrCreate()


# 1. Synthetic Dataset
data = [
    (1, 101, 200, -10, "2023-11-15"),
    (2, 102, 500, -20, "2023-11-20"),
    (3, 103, 100, -5,  "2023-12-10"),
    (4, 101, 200, 10,  "2023-11-25"),  # fault row
    (5, 104, 300, -15, "2023-12-05")
]
cols = ["sale_id", "item_id", "price", "discount", "date"]
df = spark.createDataFrame(data, cols)


# 2. Branch Profiling → Path Vectors
def branch_profile(df):
    """Assign each row a path vector for branches."""
    profiled = df.withColumn("b_nov", when(col("date").substr(6,2)=="11",1).otherwise(0)) \
                .withColumn("b_high", when(col("price")>250,1).otherwise(0)) \
                .withColumn("b_itemA", when(col("item_id")==101,1).otherwise(0))
    return profiled

profiled_df = branch_profile(df)

def get_path_vector(row):
    """Convert branch columns into a binary vector tuple."""
    return (row.b_nov, row.b_high, row.b_itemA)


# 3. Seed Selection (bitwise OR logic)
def select_seeds(df):
    """Keep rows that add new coverage."""
    seen = set()
    seeds = []
    for row in df.collect():
        vec = get_path_vector(row)
        if vec not in seen:
            seeds.append(row)
            seen.add(vec)
    return seeds

seed_rows = select_seeds(profiled_df)


# 4. Interleaving Mutation
def interleave_mutation(seeds):
    """Column-wise interleaving from donor rows to create new rows."""
    mutated = []
    for row in seeds:
        row_dict = row.asDict()
        donor = random.choice(seeds).asDict()

        # flip price branch if possible
        if row_dict["price"] <= 250 and donor["price"] > 250:
            row_dict["price"] = donor["price"]
        # flip item branch if possible
        if row_dict["item_id"] != 101 and donor["item_id"] == 101:
            row_dict["item_id"] = donor["item_id"]
        # always fix discount if invalid
        if row_dict["discount"] > 0:
            row_dict["discount"] = donor["discount"]

        mutated.append(row_dict)
    return spark.createDataFrame(mutated)

mutated_df = interleave_mutation(seed_rows)
mutated_df = branch_profile(mutated_df)  # re-profile after mutation

# 5. Evaluation Metrics
def evaluate(df):
    total = df.count()
    coverage = len({get_path_vector(r) for r in df.collect()})
    faults = df.filter(col("discount") > 0).count()
    natural = df.filter(col("price").between(50,1000)).count() / total * 100
    return coverage, faults, natural

coverage, faults, natural = evaluate(mutated_df)


# 6. Results
print("\n=== NaturalFuzz ===")
print(f"Coverage: {coverage}")
print(f"Faults: {faults}")
print(f"Naturalness: {natural:.1f}%")

print("\nFinal Mutated Data:")
mutated_df.show()

spark.stop()


=== Evaluation Metrics ===

--------------------------------------------------
Tool         | Coverage | Faults | Naturalness 
--------------------------------------------------
NaturalFuzz  | 3        | 1      | 80.0%
Jazzer       | 4        | 4      | 20.0%
BigFuzz      | 3        | 1      | 0.0%

=== Sample Faults Detected ===

NaturalFuzz:
+-----------------+-------------+---------------+----------+--------+-------+-----+-------+
|branch_high_value|branch_item_A|branch_november|date      |discount|item_id|price|sale_id|
+-----------------+-------------+---------------+----------+--------+-------+-----+-------+
|0                |1            |1              |2023-11-25|10      |101    |200  |4      |
+-----------------+-------------+---------------+----------+--------+-------+-----+-------+


Jazzer:
+---------+--------+-------+-----+-------+---------------+-----------------+-------------+
|date     |discount|item_id|price|sale_id|branch_november|branch_high_value|branch_item_A|
+-